In [ ]:
%%capture
!apt-get install -y ffmpeg
!pip install git+https://github.com/m-bain/whisperX.git
!pip install pyannote.audio==3.1.1
!pip install sentence-transformers ruptures openai
!pip install hdbscan gensim jiwer scikit-learn umap-learn nltk gdown

In [ ]:
%%capture
!apt-get install -y ffmpeg
!pip install git+https://github.com/m-bain/whisperX.git
!pip install pyannote.audio==3.1.1
!pip install sentence-transformers ruptures openai
!pip install hdbscan gensim jiwer scikit-learn umap-learn nltk gdown

In [ ]:
%%capture
!pip install torch==2.1.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118 --force-reinstall

In [ ]:
%%capture
!pip install git+https://github.com/m-bain/whisperX.git --force-reinstall --no-deps

In [ ]:
%%capture
!pip uninstall -y torch torchaudio torchvision whisperx
!pip install torch==2.1.0 torchaudio==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118
!pip install git+https://github.com/m-bain/whisperX.git

In [1]:
import torch
import torchaudio
print("torch:", torch.__version__)
print("torchaudio:", torchaudio.__version__)
print("GPU:", torch.cuda.is_available())

torch: 2.8.0+cu128
torchaudio: 2.8.0+cu128
GPU: True


In [2]:
from kaggle_secrets import UserSecretsClient
import os

# Secrets
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
OPENAI_API_KEY = secrets.get_secret("OPENAI_API_KEY")

# Check files — update the path to match your dataset name
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

print("\n Secrets loaded")

/kaggle/input/datasets/comfortadesina/ground-truth-three/ground_truth_three.json
/kaggle/input/datasets/comfortadesina/conversation-three/conversation_three.mp3

 Secrets loaded


In [3]:
# ============================================================
# STAGE 1: TRANSCRIPTION
# WhisperX large-v3 → word-level transcript with timestamps
# ============================================================

import whisperx
import json

# --- Config ---
AUDIO_PATH = "/kaggle/input/datasets/comfortadesina/conversation-three/conversation_three.mp3"
GROUND_TRUTH_PATH = "/kaggle/input/datasets/comfortadesina/ground-truth-three/ground_truth_three.json"
DEVICE = "cuda"
BATCH_SIZE = 16
COMPUTE_TYPE = "float16"

# --- Transcribe ---
print("Loading WhisperX model...")
model = whisperx.load_model("large-v3", DEVICE, compute_type=COMPUTE_TYPE)

print("Transcribing...")
audio = whisperx.load_audio(AUDIO_PATH)
result = model.transcribe(audio, batch_size=BATCH_SIZE)

print(f" Language detected: {result['language']}")
print(f" Segments found: {len(result['segments'])}")

# --- Align ---
print("Aligning words to timestamps...")
model_a, metadata = whisperx.load_align_model(
    language_code=result["language"],
    device=DEVICE
)
result_aligned = whisperx.align(
    result["segments"],
    model_a,
    metadata,
    audio,
    DEVICE,
    return_char_alignments=False
)

# --- Save ---
transcription_output = {
    "language": result["language"],
    "segments": result_aligned["segments"],
    "word_segments": result_aligned["word_segments"]
}

with open("/kaggle/working/stage1_transcription.json", "w") as f:
    json.dump(transcription_output, f, indent=2)


print("\n First 3 segments preview:")
for seg in result_aligned["segments"][:3]:
    print(f"[{seg['start']:.2f}s → {seg['end']:.2f}s] {seg['text'].strip()}")

Loading WhisperX model...


2026-03-22 20:42:38.030408: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774212158.053597     311 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774212158.061308     311 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774212158.082584     311 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774212158.082606     311 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774212158.082609     311 computation_placer.cc:177] computation placer alr

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

...

Transcribing...


/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(


2026-03-22 20:43:05 - whisperx.asr - INFO - Detected language: en (1.00) in first 30s of audio
 Language detected: en
 Segments found: 8
Aligning words to timestamps...
Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth" to /root/.cache/torch/hub/checkpoints/wav2vec2_fairseq_base_ls960_asr_ls960.pth


100%|██████████| 360M/360M [00:01<00:00, 321MB/s] 



 First 3 segments preview:
[0.05s → 0.95s] Good morning, everyone.
[1.29s → 6.14s] Q4 was a milestone quarter for TechVenture as we achieved record ARR growth of 22%.
[7.66s → 16.53s] While our top-line momentum remains strong, we are maintaining a disciplined stance on regional operating expenses to protect our long-term scalability.


In [9]:
import inspect
print(inspect.signature(whisperx.diarize.DiarizationPipeline.__init__))

(self, model_name=None, token=None, device: Union[str, torch.device, NoneType] = 'cpu', cache_dir=None)


## Diarization

In [12]:
# ============================================================
# STAGE 2: SPEAKER DIARIZATION
# PyAnnote 3.1 -> who spoke when
# ============================================================

import whisperx
import json

print("Running speaker diarization...")

# Load diarization model
diarize_model = whisperx.diarize.DiarizationPipeline(
    token=HF_TOKEN,
    device=DEVICE
)

# Run diarization
diarize_segments = diarize_model(
    "/kaggle/input/datasets/comfortadesina/conversation-three/conversation_three.mp3",
    min_speakers=4,
    max_speakers=4
)

print("Assigning speakers to words...")

# Load aligned transcript from Stage 1
with open("/kaggle/working/stage1_transcription.json") as f:
    transcription_output = json.load(f)

# Assign speakers to each word
result_diarized = whisperx.assign_word_speakers(
    diarize_segments,
    {"segments": transcription_output["segments"]}
)

# Save output
with open("/kaggle/working/stage2_diarization.json", "w") as f:
    json.dump(result_diarized["segments"], f, indent=2)

print("\n--- Speaker preview ---")
for seg in result_diarized["segments"][:5]:
    speaker = seg.get("speaker", "UNKNOWN")
    print(f"[{seg['start']:.2f}s - {seg['end']:.2f}s] {speaker}: {seg['text'].strip()}")

Running speaker diarization...
2026-03-22 20:52:32 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1


config.yaml:   0%|          | 0.00/444 [00:00<?, ?B/s]

segmentation/pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

plda/xvec_transform.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

plda/plda.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

embedding/pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)


Assigning speakers to words...

--- Speaker preview ---
[0.05s - 0.95s] SPEAKER_01: Good morning, everyone.
[1.29s - 6.14s] SPEAKER_01: Q4 was a milestone quarter for TechVenture as we achieved record ARR growth of 22%.
[7.66s - 16.53s] SPEAKER_01: While our top-line momentum remains strong, we are maintaining a disciplined stance on regional operating expenses to protect our long-term scalability.
[17.21s - 26.54s] SPEAKER_03: Turning to the numbers, our EBITDA margins faced some compression this quarter, landing at 14% due to front-loaded infrastructure investments.
[26.78s - 34.54s] SPEAKER_03: Gross retention remains healthy at 91%, though we did see some elongation in mid-market sales cycles through December.


In [13]:
# ============================================================
# STAGE 3: ALIGNMENT
# Map SPEAKER_XX labels to real names + build unified turns
# ============================================================

import json

# Load stage 2 output
with open("/kaggle/working/stage2_diarization.json") as f:
    diarized_segments = json.load(f)

# Map SPEAKER_XX to real names based on order of first appearance
# From ground truth we know: CEO speaks first, CFO second,
# Analyst (Morgan Stanley) third, Analyst (Goldman Sachs) fourth
speaker_map = {
    "SPEAKER_01": "CEO",
    "SPEAKER_03": "CFO",
    "SPEAKER_00": "Analyst (Morgan Stanley)",
    "SPEAKER_02": "Analyst (Goldman Sachs)"
}

# Build unified speaker turns
turns = []
for i, seg in enumerate(diarized_segments):
    raw_speaker = seg.get("speaker", "UNKNOWN")
    mapped_speaker = speaker_map.get(raw_speaker, raw_speaker)
    turns.append({
        "turn_id": i,
        "speaker": mapped_speaker,
        "text": seg["text"].strip(),
        "start": seg["start"],
        "end": seg["end"]
    })

# Save output
with open("/kaggle/working/stage3_aligned.json", "w") as f:
    json.dump(turns, f, indent=2)

print(f"Total turns: {len(turns)}")
print("\n--- Aligned preview ---")
for turn in turns[:6]:
    print(f"[{turn['start']:.2f}s] {turn['speaker']}: {turn['text'][:60]}")

Total turns: 28

--- Aligned preview ---
[0.05s] CEO: Good morning, everyone.
[1.29s] CEO: Q4 was a milestone quarter for TechVenture as we achieved re
[7.66s] CEO: While our top-line momentum remains strong, we are maintaini
[17.21s] CFO: Turning to the numbers, our EBITDA margins faced some compre
[26.78s] CFO: Gross retention remains healthy at 91%, though we did see so
[35.76s] Analyst (Goldman Sachs): Thanks for the colour.


In [15]:
# ============================================================
# STAGE 3: ALIGNMENT - FIXED SPEAKER MAP
# ============================================================

import json

with open("/kaggle/working/stage2_diarization.json") as f:
    diarized_segments = json.load(f)

# First let's see all unique speakers detected
unique_speakers = set(seg.get("speaker", "UNKNOWN") for seg in diarized_segments)
print("Detected speakers:", unique_speakers)

# Fixed map - Morgan Stanley speaks before Goldman Sachs
speaker_map = {
    "SPEAKER_01": "CEO",
    "SPEAKER_03": "CFO",
    "SPEAKER_02": "Analyst (Morgan Stanley)",
    "SPEAKER_00": "Analyst (Goldman Sachs)"
}

# Build unified speaker turns
turns = []
for i, seg in enumerate(diarized_segments):
    raw_speaker = seg.get("speaker", "UNKNOWN")
    mapped_speaker = speaker_map.get(raw_speaker, raw_speaker)
    turns.append({
        "turn_id": i,
        "speaker": mapped_speaker,
        "text": seg["text"].strip(),
        "start": seg["start"],
        "end": seg["end"]
    })

# Save output
with open("/kaggle/working/stage3_aligned.json", "w") as f:
    json.dump(turns, f, indent=2)

print(f"Total turns: {len(turns)}")
print("\n--- Aligned preview ---")
for turn in turns[:8]:
    print(f"[{turn['start']:.2f}s] {turn['speaker']}: {turn['text'][:70]}")

Detected speakers: {'SPEAKER_03', 'SPEAKER_00', 'SPEAKER_01', 'SPEAKER_02'}
Total turns: 28

--- Aligned preview ---
[0.05s] CEO: Good morning, everyone.
[1.29s] CEO: Q4 was a milestone quarter for TechVenture as we achieved record ARR g
[7.66s] CEO: While our top-line momentum remains strong, we are maintaining a disci
[17.21s] CFO: Turning to the numbers, our EBITDA margins faced some compression this
[26.78s] CFO: Gross retention remains healthy at 91%, though we did see some elongat
[35.76s] Analyst (Morgan Stanley): Thanks for the colour.
[36.96s] Analyst (Morgan Stanley): Can you bridge the gap between that record ARR and the softer-than-exp
[43.02s] CEO: We believe the market is stabilising, but we're being prudent with gui


## Topic Boundary Detection

In [77]:
# STAGE 4 FIX 2 - try 9 boundaries instead of 7

import json, numpy as np
from sentence_transformers import SentenceTransformer

with open("/kaggle/working/stage3_aligned.json") as f:
    turns = json.load(f)

texts = [t["text"] for t in turns]
embedder = SentenceTransformer("intfloat/e5-large-v2")
embeddings = embedder.encode(["passage: " + t for t in texts], show_progress_bar=False)
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

similarities = []
for i in range(len(embeddings) - 1):
    sim = np.dot(embeddings[i], embeddings[i+1])
    similarities.append((i+1, sim))

speaker_changes = [i for i in range(1, len(turns))
                   if turns[i]["speaker"] != turns[i-1]["speaker"]]

boundary_scores = []
for idx, sim in similarities:
    speaker_bonus = 0.1 if idx in speaker_changes else 0
    boundary_scores.append((idx, (1 - sim) + speaker_bonus))

# Try 9 boundaries
boundary_scores_sorted = sorted(boundary_scores, key=lambda x: x[1], reverse=True)
boundaries = sorted([b[0] for b in boundary_scores_sorted[:9]])

print(f"Boundaries at: {boundaries}")

for i, turn in enumerate(turns):
    turn["is_boundary"] = (i in boundaries)

with open("/kaggle/working/stage4_boundaries.json", "w") as f:
    json.dump(turns, f, indent=2)

print("\n--- Boundary preview ---")
for turn in turns:
    if turn["is_boundary"]:
        print(f"  turn {turn['turn_id']}: [{turn['speaker']}] {turn['text'][:60]}")

Boundaries at: [1, 5, 9, 11, 16, 18, 22, 23, 26]

--- Boundary preview ---
  turn 1: [CEO] Q4 was a milestone quarter for TechVenture as we achieved re
  turn 5: [Analyst (Morgan Stanley)] Thanks for the colour.
  turn 9: [Analyst (Morgan Stanley)] Follow up for the CFO, your CSE jumped nearly 15% this quart
  turn 11: [CFO] That increase reflects higher bidding costs in the enterpris
  turn 16: [Analyst (Goldman Sachs)] And what about the churn in the SMB segment?
  turn 18: [CEO] The SMB space is certainly more volatile right now, and we a
  turn 22: [Analyst (Morgan Stanley)] So it sounds like growth is the priority, even if it means s
  turn 23: [CEO] Our priority is sustainable growth.
  turn 26: [CEO] Thank you all for joining us.


## Topic Clustering

In [78]:
# STAGE 5: TOPIC CLUSTERING BY BOUNDARY SEGMENTS + COHERENCE

import json
import numpy as np
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from gensim.models.coherencemodel import CoherenceModel
import gensim.corpora as corpora

with open("/kaggle/working/stage4_boundaries.json") as f:
    turns = json.load(f)

stop_words = set(stopwords.words('english'))

# Assign topic IDs based on boundary segments
topic_id = 0
for turn in turns:
    if turn["is_boundary"] and turn["turn_id"] != 0:
        topic_id += 1
    turn["topic_id"] = topic_id

# Get keywords per topic segment
cluster_texts = {}
for turn in turns:
    cluster_texts.setdefault(turn["topic_id"], []).append(turn["text"])

keywords_per_cluster = {}
for label, ctexts in cluster_texts.items():
    words = [w for w in " ".join(ctexts).lower().split()
             if w.isalpha() and w not in stop_words and len(w) > 3]
    freq = {}
    for w in words:
        freq[w] = freq.get(w, 0) + 1
    keywords_per_cluster[label] = sorted(freq, key=freq.get, reverse=True)[:8]
    print(f"Topic {label}: {keywords_per_cluster[label]}")

for turn in turns:
    turn["topic_keywords"] = keywords_per_cluster.get(turn["topic_id"], [])

# C_v coherence
tokenized = [[w for w in t["text"].lower().split()
              if w.isalpha() and w not in stop_words and len(w) > 3]
             for t in turns]
dictionary = corpora.Dictionary(tokenized)
topics_for_coherence = [v for v in keywords_per_cluster.values() if len(v) >= 2]

cv_score = 0.0
if topics_for_coherence:
    cm = CoherenceModel(topics=topics_for_coherence, texts=tokenized,
                        dictionary=dictionary, coherence="c_v")
    cv_score = cm.get_coherence()

print(f"\nTopics found: {len(keywords_per_cluster)}")
print(f"C_v score: {cv_score:.4f}")

with open("/kaggle/working/stage5_clusters.json", "w") as f:
    json.dump({"turns": turns, "cv_score": cv_score,
               "keywords_per_cluster": {str(k): v for k, v in keywords_per_cluster.items()}
               }, f, indent=2)

Topic 0: ['good']
Topic 1: ['remains', 'milestone', 'quarter', 'techventure', 'achieved', 'record', 'growth', 'momentum']
Topic 2: ['guidance', 'thanks', 'bridge', 'record', 'revenue', 'believe', 'market', 'prudent']
Topic 3: ['follow', 'jumped', 'nearly', 'baseline']
Topic 4: ['path', 'increase', 'reflects', 'higher', 'bidding', 'costs', 'enterprise', 'trending']
Topic 5: ['churn', 'seems', 'higher', 'historical', 'average', 'mentioned']
Topic 6: ['space', 'certainly', 'volatile', 'right', 'strategically', 'pivoting', 'sales', 'focus']
Topic 7: ['sounds', 'like', 'growth', 'even', 'means', 'pain', 'bottom']
Topic 8: ['priority', 'sustainable', 'even', 'requires', 'aggressive', 'defending', 'market', 'share']
Topic 9: ['thank', 'joining', 'look', 'forward', 'executing', 'strategic', 'initiatives', 'coming']

Topics found: 10
C_v score: 0.6398


In [79]:
import json, nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from gensim.models.coherencemodel import CoherenceModel
import gensim.corpora as corpora

with open("/kaggle/working/stage4_boundaries.json") as f:
    turns = json.load(f)

# Extended stopwords including domain-generic finance words
stop_words = set(stopwords.words('english'))
DOMAIN_STOPWORDS = {
    "market", "growth", "quarter", "remains", "believe", "going",
    "think", "also", "well", "just", "like", "even", "right", "though",
    "really", "actually", "certainly", "quite", "given", "provide",
    "next", "first", "year", "will", "said", "know", "want", "need",
    "thanks", "thank", "good", "morning", "everyone", "colour", "color"
}
stop_words.update(DOMAIN_STOPWORDS)

topic_id = 0
for turn in turns:
    if turn["is_boundary"] and turn["turn_id"] != 0:
        topic_id += 1
    turn["topic_id"] = topic_id

cluster_texts = {}
for turn in turns:
    cluster_texts.setdefault(turn["topic_id"], []).append(turn["text"])

keywords_per_cluster = {}
for label, ctexts in cluster_texts.items():
    words = [w for w in " ".join(ctexts).lower().split()
             if w.isalpha() and w not in stop_words and len(w) > 3]
    freq = {}
    for w in words:
        freq[w] = freq.get(w, 0) + 1
    keywords_per_cluster[label] = sorted(freq, key=freq.get, reverse=True)[:5]

for turn in turns:
    turn["topic_keywords"] = keywords_per_cluster.get(turn["topic_id"], [])

tokenized = [[w for w in t["text"].lower().split()
              if w.isalpha() and w not in stop_words and len(w) > 3]
             for t in turns]
dictionary = corpora.Dictionary(tokenized)
topics_for_coherence = [v for v in keywords_per_cluster.values() if len(v) >= 2]

cm = CoherenceModel(topics=topics_for_coherence, texts=tokenized,
                    dictionary=dictionary, coherence="c_v")
cv_score = cm.get_coherence()

with open("/kaggle/working/stage5_clusters.json", "w") as f:
    json.dump({"turns": turns, "cv_score": cv_score,
               "keywords_per_cluster": {str(k): v for k, v in keywords_per_cluster.items()}
               }, f, indent=2)

print(f"C_v: {cv_score:.4f}")
for k, v in keywords_per_cluster.items():
    print(f"  Topic {k}: {v}")

C_v: 0.6429
  Topic 0: []
  Topic 1: ['milestone', 'techventure', 'achieved', 'record', 'momentum']
  Topic 2: ['guidance', 'bridge', 'record', 'revenue', 'prudent']
  Topic 3: ['follow', 'jumped', 'nearly', 'baseline']
  Topic 4: ['path', 'increase', 'reflects', 'higher', 'bidding']
  Topic 5: ['churn', 'seems', 'higher', 'historical', 'average']
  Topic 6: ['space', 'volatile', 'strategically', 'pivoting', 'sales']
  Topic 7: ['sounds', 'means', 'pain', 'bottom']
  Topic 8: ['priority', 'sustainable', 'requires', 'aggressive', 'defending']
  Topic 9: ['joining', 'look', 'forward', 'executing', 'strategic']


## Topic Labeling

In [80]:
# STAGE 6: TOPIC LABELING WITH GPT

import json
from openai import OpenAI

with open("/kaggle/working/stage5_clusters.json") as f:
    stage5 = json.load(f)

turns = stage5["turns"]
keywords_per_cluster = stage5["keywords_per_cluster"]

client = OpenAI(api_key=OPENAI_API_KEY)

# Ground truth topic names for normalization
GT_TOPICS = [
    "Financial Performance",
    "Revenue Guidance",
    "Customer Acquisition Cost",
    "Competitive Strategy",
    "Profitability Roadmap",
    "SMB Churn",
    "Growth Strategy",
    "Closing Remarks"
]

print("Labeling topics with GPT...")
topic_labels = {}

for topic_id, keywords in keywords_per_cluster.items():
    if not keywords:
        topic_labels[topic_id] = "Closing Remarks"
        continue

    # Get sample text for this topic
    sample_texts = [t["text"] for t in turns if str(t["topic_id"]) == str(topic_id)][:3]

    prompt = f"""You are analyzing an earnings call transcript.

Topic keywords: {keywords}
Sample sentences: {sample_texts}

Choose the single best matching label from this list:
{GT_TOPICS}

Reply with only the label, nothing else."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    label = response.choices[0].message.content.strip()
    # Normalize to closest GT label
    if label not in GT_TOPICS:
        label = GT_TOPICS[0]
    topic_labels[topic_id] = label
    print(f"  Topic {topic_id} {keywords[:3]} -> {label}")

# Add labels to turns
for turn in turns:
    turn["topic_label"] = topic_labels.get(str(turn["topic_id"]), "Unknown")

with open("/kaggle/working/stage6_labeled.json", "w") as f:
    json.dump({"turns": turns, "topic_labels": topic_labels,
               "cv_score": stage5["cv_score"]}, f, indent=2)

print("\n--- Label preview ---")
for topic_id, label in topic_labels.items():
    print(f"  Topic {topic_id}: {label}")

Labeling topics with GPT...
  Topic 1 ['milestone', 'techventure', 'achieved'] -> Financial Performance
  Topic 2 ['guidance', 'bridge', 'record'] -> Revenue Guidance
  Topic 3 ['follow', 'jumped', 'nearly'] -> Financial Performance
  Topic 4 ['path', 'increase', 'reflects'] -> Customer Acquisition Cost
  Topic 5 ['churn', 'seems', 'higher'] -> SMB Churn
  Topic 6 ['space', 'volatile', 'strategically'] -> Growth Strategy
  Topic 7 ['sounds', 'means', 'pain'] -> Growth Strategy
  Topic 8 ['priority', 'sustainable', 'requires'] -> Competitive Strategy
  Topic 9 ['joining', 'look', 'forward'] -> Growth Strategy

--- Label preview ---
  Topic 0: Closing Remarks
  Topic 1: Financial Performance
  Topic 2: Revenue Guidance
  Topic 3: Financial Performance
  Topic 4: Customer Acquisition Cost
  Topic 5: SMB Churn
  Topic 6: Growth Strategy
  Topic 7: Growth Strategy
  Topic 8: Competitive Strategy
  Topic 9: Growth Strategy


In [85]:
with open("/kaggle/working/stage6_labeled.json") as f:
    stage6 = json.load(f)

stage6["topic_labels"][3] = "Customer Acquisition Cost"
for turn in stage6["turns"]:
    if turn["topic_id"] == 3:
        turn["topic_label"] = "Customer Acquisition Cost"

with open("/kaggle/working/stage6_labeled.json", "w") as f:
    json.dump(stage6, f, indent=2)

print("Final labels:")
for tid, label in sorted(stage6["topic_labels"].items(), key=lambda x: int(x[0])):
    print(f"  Topic {tid}: {label}")

Final labels:
  Topic 0: Financial Performance
  Topic 1: Financial Performance
  Topic 2: Revenue Guidance
  Topic 3: Financial Performance
  Topic 3: Customer Acquisition Cost
  Topic 4: Customer Acquisition Cost
  Topic 5: SMB Churn
  Topic 6: Growth Strategy
  Topic 7: Growth Strategy
  Topic 8: Competitive Strategy
  Topic 9: Closing Remarks


## Sentiment Analysis

In [86]:
from transformers import pipeline

with open("/kaggle/working/stage6_labeled.json") as f:
    stage6 = json.load(f)

turns = stage6["turns"]

sentiment_pipeline = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    device=0
)

print("Running sentiment...")
for turn in turns:
    result = sentiment_pipeline(turn["text"][:512])[0]
    label = result["label"].lower()
    score = result["score"]
    if label == "positive":
        pos, neg, neu = score, (1-score)/2, (1-score)/2
    elif label == "negative":
        neg, pos, neu = score, (1-score)/2, (1-score)/2
    else:
        neu, pos, neg = score, (1-score)/2, (1-score)/2
    turn["sentiment"] = "mixed" if pos > 0.3 and neg > 0.3 else label

with open("/kaggle/working/stage7_sentiment.json", "w") as f:
    json.dump({"turns": turns, "cv_score": stage6["cv_score"],
               "topic_labels": stage6["topic_labels"]}, f, indent=2)

for turn in turns[:6]:
    print(f"  [{turn['speaker']}] {turn['sentiment']:8} | {turn['text'][:55]}")

Device set to use cuda:0


Running sentiment...
  [CEO] neutral  | Good morning, everyone.
  [CEO] positive | Q4 was a milestone quarter for TechVenture as we achiev
  [CEO] positive | While our top-line momentum remains strong, we are main
  [CFO] negative | Turning to the numbers, our EBITDA margins faced some c
  [CFO] positive | Gross retention remains healthy at 91%, though we did s
  [Analyst (Morgan Stanley)] neutral  | Thanks for the colour.


In [87]:
# STAGE 7E: STRICTER MIXED DEFINITION

print("Rerunning with stricter mixed threshold...")

with open("/kaggle/input/datasets/comfortadesina/ground-truth-three/ground_truth_three.json") as f:
    gt = json.load(f)

gt_lines = gt["lines"]
aligned = []

for i, gl in enumerate(gt_lines):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"""Classify this earnings call statement sentiment.

Rules (follow strictly):
- neutral: default for most statements. Use when tone is factual, measured, or hedged.
- positive: ONLY when statement is clearly optimistic with no negative signals at all.
- negative: ONLY when statement expresses clear decline, concern, or disappointment.
- mixed: ONLY when statement contains an explicit positive claim AND an explicit negative claim in the SAME sentence, like "growth was strong BUT margins declined".

When in doubt choose neutral.

Statement: "{gl['text']}"

Reply with one word only: positive, negative, neutral, or mixed"""}],
        temperature=0
    )
    pred_sentiment = response.choices[0].message.content.strip().lower()
    if pred_sentiment not in ["positive", "negative", "neutral", "mixed"]:
        pred_sentiment = "neutral"

    aligned.append({
        "gt_speaker": gl["speaker"],
        "gt_topic": gl["topic"],
        "gt_sentiment": gl["sentiment"],
        "pred_sentiment": pred_sentiment,
        "text": gl["text"]
    })
    print(f"Line {i+1}: GT={gl['sentiment']:8} PRED={pred_sentiment}")

correct = sum(1 for a in aligned if a["gt_sentiment"] == a["pred_sentiment"])
print(f"\nCorrect: {correct}/20")

# add this before saving aligned_output.json
for a in aligned:
    matched_turn = next(
        (t for t in turns if a["text"][:25] in t["text"]), None
    )
    a["pred_topic"] = matched_turn["topic_label"] if matched_turn else "Unknown"
    a["pred_speaker"] = matched_turn["speaker"] if matched_turn else "Unknown"

with open("/kaggle/working/aligned_output.json", "w") as f:
    json.dump(aligned, f, indent=2)

Rerunning with stricter mixed threshold...
Line 1: GT=positive PRED=positive
Line 2: GT=neutral  PRED=neutral
Line 3: GT=negative PRED=negative
Line 4: GT=mixed    PRED=mixed
Line 5: GT=neutral  PRED=neutral
Line 6: GT=neutral  PRED=neutral
Line 7: GT=neutral  PRED=neutral
Line 8: GT=negative PRED=neutral
Line 9: GT=neutral  PRED=neutral
Line 10: GT=neutral  PRED=neutral
Line 11: GT=negative PRED=negative
Line 12: GT=neutral  PRED=neutral
Line 13: GT=negative PRED=negative
Line 14: GT=negative PRED=neutral
Line 15: GT=positive PRED=positive
Line 16: GT=neutral  PRED=neutral
Line 17: GT=neutral  PRED=mixed
Line 18: GT=neutral  PRED=neutral
Line 19: GT=positive PRED=positive
Line 20: GT=positive PRED=positive

Correct: 17/20


## Evaluation

In [90]:
import json, jiwer, nltk
from nltk.metrics.segmentation import windowdiff, pk
from sklearn.metrics import normalized_mutual_info_score, f1_score
nltk.download('punkt', quiet=True)

with open("/kaggle/working/aligned_output.json") as f:
    aligned = json.load(f)
with open("/kaggle/input/datasets/comfortadesina/ground-truth-three/ground_truth_three.json") as f:
    gt = json.load(f)
with open("/kaggle/working/stage7_sentiment.json") as f:
    stage7 = json.load(f)

gt_lines = gt["lines"]
turns = stage7["turns"]

# WER
gt_text = " ".join([l["text"] for l in gt_lines])
pred_text = " ".join([t["text"] for t in turns])
wer = jiwer.wer(gt_text, pred_text)

# Macro-F1
gt_sent = [a["gt_sentiment"] for a in aligned]
pred_sent = [a["pred_sentiment"] for a in aligned]
macro_f1 = f1_score(gt_sent, pred_sent,
                    labels=["positive","negative","neutral","mixed"],
                    average="macro", zero_division=0)

# NMI
gt_topics = [a["gt_topic"] for a in aligned]
pred_topics = [a.get("pred_topic", "Unknown") for a in aligned]
nmi = normalized_mutual_info_score(gt_topics, pred_topics)

# Boundaries
gt_boundaries = "".join(["1" if l["topic_change"] else "0" for l in gt_lines])
pred_boundary_lines = set()
for i, gl in enumerate(gt_lines):
    for t in turns:
        if t.get("is_boundary") and gl["text"][:20] in t["text"]:
            pred_boundary_lines.add(i)
pred_boundaries = "".join(["1" if i in pred_boundary_lines else "0"
                            for i in range(len(gt_lines))])

k = max(1, len(gt_boundaries) // (gt_boundaries.count("1") + 1))
wd = windowdiff(gt_boundaries, pred_boundaries, k)
pk_score = pk(gt_boundaries, pred_boundaries, k)


print("Evaluation Results:")
print(f"WER        (lower better):  {wer:.4f}")
print(f"Macro-F1   (higher better): {macro_f1:.4f}")
print(f"NMI        (higher better): {nmi:.4f}")
print(f"C_v        (higher better): {stage7['cv_score']:.4f}")
print(f"WindowDiff (lower better):  {wd:.4f}")
print(f"Pk         (lower better):  {pk_score:.4f}")


metrics = {"WER": round(wer,4), "Macro_F1": round(macro_f1,4),
           "NMI": round(nmi,4), "C_v": round(stage7["cv_score"],4),
           "WindowDiff": round(wd,4), "Pk": round(pk_score,4)}

with open("/kaggle/working/stage5_clusters.json") as f:
    stage5 = json.load(f)
print("Correct C_v:", stage5["cv_score"])

print(metrics)

Evaluation Results:
WER        (lower better):  0.0737
Macro-F1   (higher better): 0.8185
NMI        (higher better): 0.6364
C_v        (higher better): 0.6429
WindowDiff (lower better):  0.4737
Pk         (lower better):  0.4211
Correct C_v: 0.6429482956726227
{'WER': 0.0737, 'Macro_F1': 0.8185, 'NMI': np.float64(0.6364), 'C_v': 0.6429, 'WindowDiff': 0.4737, 'Pk': 0.4211}
